# Diffuison Policy Training

The goal of this notebook is to train the base Diffusion Policy (DP) that later gets fine-tuned using DynaGuide or DSRL.</br>
Training is done using [robomimic](https://robomimic.github.io/), operating on the `square_d0` dataset obtained in **Dataset_Prepreation.ipynb**.

This notebook expects that you have already successfully obtained a suitable dataset including train &amp; val filter keys (see **Dataset_Prepreation.ipynb**).


First activate your conda environment.

## 1. Config

The training config `configs/base_diffusion_policy/diffusion_policy_image_square_d0_ddim.json` defines the policy (image-based Diffusion Policy with DDIM sampling) and the training run. Carefully read through it before starting a run.</br>

Notable settings:
- **Observations**: `robot0_eef_pos`, `robot0_eef_quat`, `robot0_gripper_qpos` (low-dim) plus `agentview_image` and `robot0_eye_in_hand_image` (rgb, via a ResNet18 + SpatialSoftmax encoder). Your dataset needs to contain these keys.
- **Epochs / batch size**: `num_epochs: 5000`, `batch_size: 64`.
- **Checkpointing**: a checkpoint is saved every 10 epochs (`experiment.save.every_n_epochs`).
- **Rollouts**: 10 evaluation rollouts are run periodically during training (`experiment.rollout`) to track success rate.

There is also `configs/base_diffusion_policy/diffusion_policy_image_square_d0_ddim_augmentation.json`, which additionally enables image augmentation. Use this variant if you trained (or want to train) on the augmented real-world-deployment dataset from **Dataset_Prepreation.ipynb**.</br>

## 2. Train

Now start training, pointing `--dataset` at your prepared `square_d0.hdf5` (with train/valid filter keys):

```bash
python dyna_scripts/train_base_policy.py \
    --config configs/base_diffusion_policy/diffusion_policy_image_square_d0_ddim.json \
    --dataset dataset/square_d0.hdf5 \
    --name base_policy_square_d0 \
    --output results/base_policy_square_d0
```

Checkpoints are written to `results/base_policy_square_d0/TIMESTAMP/models/model_epoch_N.pth`.</br>
For downstream use with **DynaGuide.ipynb** / **DSRL.ipynb**, copy the checkpoint you want into `dataset/` and give it a descriptive name (e.g. `ep_300_square_d0.pth`).</br>
Use `--output` to override the output folder, or `--name` to override the experiment name, if needed.

Set `ulimit -n 65536` to avoid training exceptions.
Or for non root users `sudo prlimit --nofile=65536:65536 --pid=$$`

## 3. Monitor Training

Training logs (loss, rollout success rate, etc.) are written as tensorboard logs alongside the checkpoints. In a separate terminal run:

```bash
tensorboard --logdir results/base_policy_square_d0
```

## 4. Evaluate a Checkpoint

Unlike the rollouts during training, this replays the same fixed validation states for every checkpoint, so success rates are actually comparable:

```bash
python dyna_scripts/run_trained_agent_in_order.py \
    --agent  results/base_policy_square_d0/TIMESTAMP/models/model_epoch_XXX.pth \
    --horizon 200 \
    --n_rollouts 100 \
    --eval_dataset  dataset/square_splits/square_valid.hdf5 \
    --env NutAssemblySquare \
    --seed 0
```

Add `--video_path out.mp4` or `--dataset_path out.hdf5` to save the rollouts.

---
After having obtaind a suitable diffusion policy checkpoint, continue with [DynaGuide](DynaGuide.ipynb).